In [25]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts', 'Dates']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives', 'JioSaavn']


In [27]:
from ioutils import HTMLIO, FileIO
hio = HTMLIO()
io = FileIO()
bsdata = hio.get("/Users/tgadfort/Desktop/Artist Search - Album of The Year.html")

In [ ]:
from bs4 import element

In [28]:
from lib import albumoftheyear
mio   = albumoftheyear.MusicDBIO(verbose=False)
webio = albumoftheyear.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant AlbumOfTheYear DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AlbumOfTheYear


In [33]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = Series(dtype='object') #mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [34]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Local Artist Search:       {0}".format(len(localArtists.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Search Artists:            {0}".format(searchArtists.shape[0]))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

AlbumOfTheYear Search Results
   Global Master Search:      2
   Global Master Search Data: 2
   Local Artist Search:       0
   Errors:                    0
   Search Artists:            52
   Known Summary IDs:         0


# Search For New Artists

In [16]:
mio   = albumoftheyear.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = albumoftheyear.RawWebData(debug=False)

## Find Artists To Search For

In [61]:
useMasterData = True

if useMasterData is True:
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames = mmeDF["ArtistName"].drop_duplicates()
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del mmeDF
    del pdbio

print("# {0} Search Results".format(db))
print("#   Available Names:      {0}".format(len(artistNames)))
print("#   Known Artist Names:   {0}".format(len(masterArtistsDict)))
print("#   Artist Names To Get:  {0}".format(len(artistNamesToGet)))

#   Artist Names To Get:  818852

# AlbumOfTheYear Search Results
#   Available Names:      818853
#   Known Artist Names:   13052
#   Artist Names To Get:  805801


## Download Search Data

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "9:50am")
tt = TermTime("today", "10:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue
    if searchedForErrors.get(artistName) is not None:
        continue

    try:
        response = webio.getArtistSearchData(artistName=artistName)
    except:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(10)
        continue
        
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    webio.sleep(4.0)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

Process [Getting AlbumOfTheYear ArtistIDs] Start    ==> Time Is 2022-07-08 09:03:08
========================= termTime(day=today,time=10:00pm) =========================
   ====> Terminate Time Set To 2022-07-08 22:00:00 <====
   ====> Will Terminate Process 12 Hours and 56 Minutes From Now
Searching For Phil Da Agony                                     0
Searching For Nicole Mitchell                                   2
Searching For Christian Scott                                   1
Searching For Cuco Sánchez                                     0
Searching For SOHN                                              1
Searching For Morgoth                                           2
Searching For Murder City Devils                                1
Searching For Judah & The Lion                                  1
Searching For Philanthrope                                      11
Searching For Ugly God                                          2
Searching For Agrypnie                          

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Searching For Loquillo                                          2
Searching For Gaby Baginsky                                     0
Searching For Daniel Humair                                     4
Searching For Jesper Dahlbäck                                  0
Searching For Ghostland Observatory                             1
Searching For Voxtrot                                           1
Searching For Patrick Stump                                     2
Searching For Koriass                                           2
Searching For Postmodern Jukebox                                1
Searching For Tommy Shaw                                        3
Searching For Chaka Demus                                       3
Searching For Agathodaimon                                      1
Searching For Young & Sick                                      1
Searching For Los Caminantes                                    0
Searching For Kerser                                            1
Searching 

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Searching For Mordred                                           1
Searching For Josquin Des Prés                                 0
Searching For 16                                                18
Searching For The Skyliners                                     0
Searching For Bodhi                                             3
Searching For Clifford T. Ward                                  0
Searching For Laika                                             4
Searching For Facundo Cabral                                    1
Searching For Beverley Craven                                   1
Searching For Sister Sin                                        1
Searching For Levante                                           1
Searching For Dan Ar Braz                                       1
Searching For Silent Circle                                     1
Searching For Los Melódicos                                    0
Searching For Stefano Noferini                                  0
Searching

...

Searching For Dawn of Disease                                   1
Searching For Gary's Gang                                       1
Searching For The Locust                                        5
Searching For Stella Mwangi                                     1
Searching For Funkstörung                                      0
Searching For Paper Lace                                        1
Searching For LTJ Bukem                                         3
Searching For David Dondero                                     1
Searching For Cosmo Sheldrake                                   1
Searching For Pelle Miljoona Oy                                 0
Searching For Prof                                              6
Searching For Harvey Milk                                       1
Searching For Tink                                              5
Searching For Hap Palmer                                        0
Searching For Lars Lilholt                                      0
Searching 

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Searching For George Hamilton IV                                0
Searching For Phuture                                           3
Searching For Jimmy Giuffre                                     9
Searching For The King Blues                                    2
Searching For Robin Williamson                                  1
Searching For Pinchers                                          2
Searching For Hootie & The Blowfish                             1
Searching For Tonetta                                           1
Searching For Imperial Teen                                     1
Searching For Earl Wild                                         2
Searching For Koxbox                                            0
Searching For Jordaan Mason                                     3
Searching For Gackt                                             1
Searching For Nocturnal Emissions                               1
Searching For Dan Bull                                          2
Searching 

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Searching For Quinteto Violado                                  2
Searching For Walter Beasley                                    0
Searching For Enrique Morente                                   2
Searching For Lari White                                        1
Searching For A Great Big World                                 3
Searching For Passport                                          7
Searching For Excepter                                          2
Searching For Moor Mother                                       3
Searching For Idris Elba                                        2
Searching For Blindead                                          1
Searching For Aya Nakamura                                      1
Searching For Baloji                                            1
Searching For Alabama Thunderpussy                              2
Searching For Mesita                                            4
Searching For Sigue Sigue Sputnik                               1
Searching 

In [60]:
del searchedForErrors['Against']
errors.save(data=searchedForErrors)

KeyError: 'Against'

In [55]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        searchData = {}
        for searchTerm,searchTermData in mad.items():
            if isinstance(searchTermData,dict):
                for ref,name in searchTermData.items():
                    artistID = mio.getdbid(ref)
                    if isinstance(artistID,str) and len(artistID) > 0:
                        searchData[artistID] = (ref,name)
                        #print(ref,name,artistID,searchData)
        df = DataFrame(searchData).T
        df.columns = ["Ref", "Name"]
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [56]:
mad = masterArtistsData.get()
df = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = albumoftheyear.MusicDBIO(local=False).data.getSearchArtistData()
    prevNewArtists = len(searchDF[~searchDF.index.isin(knownArtists.index)])
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF, df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF     = searchDF[~searchDF.index.duplicated(keep='first')]
    oldArtists   = searchDF[searchDF.index.isin(knownArtists.index)].shape[0]
    newArtists   = searchDF[~searchDF.index.isin(knownArtists.index)].shape[0]
    deltaArtists = searchDF.shape[0] - prevNewArtists
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("  ==> {0} Old Artists".format(oldArtists))
    print("  ==> {0} New Artists".format(newArtists))
    print("  ==> {0} Delta New Artists".format(deltaArtists))
    print("Saving Data")
    albumoftheyear.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)

Found 19286 New Artists
Found 13362 Previous Artists
Found 32648 Total Results
Found 30158 Unique Results
  ==> 0 Old Artists
  ==> 30158 New Artists
  ==> 16796 Delta New Artists
Saving Data


In [57]:
masterArtistsData.save(data={})